# Démonstration : Extraction de features
Ce notebook montre comment utiliser `src.features.extract_features` et `src.preprocessing.generate_auto_mask_from_bgr` pour extraire des caractéristiques couleur, texture (GLCM) et forme, visualiser le masque automatique et sauvegarder les résultats en JSON.

In [ ]:
# Préparation de l'environnement pour importer le package local
import sys
from pathlib import Path
repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))
import json
import cv2
import matplotlib.pyplot as plt
from src.features import extract_features
from src.preprocessing import generate_auto_mask_from_bgr
print('Imports OK')

In [ ]:
# Cherche une image d'exemple dans data/raw puis data/processed
candidates = []
raw_dir = repo_root / 'data' / 'raw'
proc_dir = repo_root / 'data' / 'processed'
if raw_dir.exists():
    candidates = [p for p in raw_dir.rglob('*') if p.suffix.lower() in ['.jpg','.jpeg','.png','.bmp','.tif','.tiff','.webp']]
if not candidates and proc_dir.exists():
    candidates = [p for p in proc_dir.rglob('*') if p.suffix.lower() in ['.jpg','.jpeg','.png','.bmp','.tif','.tiff','.webp']]
if not candidates:
    print('Aucune image trouvée dans data/raw ou data/processed. Lancez preprocessing ou ajoutez une image dans data/raw.')
else:
    img_path = candidates[0]
    print('Using image:', img_path)

In [ ]:
# Affiche l'image d'exemple
if 'img_path' in globals():
    bgr = cv2.imread(str(img_path))
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(6,6))
    plt.imshow(rgb)
    plt.axis('off')
    plt.title('Image dexemple')
    plt.show()

In [ ]:
# Génère et affiche le masque automatique (Otsu)
if 'img_path' in globals():
    mask = generate_auto_mask_from_bgr(bgr)
    plt.figure(figsize=(6,6))
    plt.imshow(mask, cmap='gray')
    plt.axis('off')
    plt.title('Masque auto (Otsu)')
    plt.show()

In [ ]:
# Extrait les features et les sauvegarde au format JSON
if 'img_path' in globals():
    vec, details = extract_features(rgb, mask=mask)
    print('Taille du vecteur de features :', vec.size)
    print('Features de forme :', details['shape'])
    out_dir = repo_root / 'notebooks_features'
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / (img_path.stem + '_features.json')
    out = {
        'image': str(img_path.relative_to(repo_root)),
        'vector_len': int(vec.size),
        'shape': details['shape']
    }
    out_path.write_text(json.dumps(out, indent=2), encoding='utf-8')
    print('Saved features to', out_path)

**Étapes suivantes :**
- Pour traiter tout le dataset et sauvegarder les features, exécutez :
  `python -m src.preprocessing --input-dir data/raw --output-dir data/processed --size 224 224 --extract-features --auto-mask`
- Les JSON de features seront enregistrés dans `notebooks_features/` (ou `--features-dir` si spécifié).
- Vous pouvez ensuite charger ces JSONs dans un `pandas.DataFrame` pour entraîner un classifieur.